# 01 — Injection functions: build and test

Tests the three fault-injection functions (ground truth for the modules). Functions live in `src/injection.py`.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))
import numpy as np, pandas as pd
from injection import inject_anomalies, inject_schema_drift, inject_missing_values

In [ ]:
DATA_PATH = '../data/HI-Small_Trans.csv'   # <-- set to your HPC path
df = pd.read_csv(DATA_PATH, nrows=100_000)
print('shape:', df.shape); print('columns:', list(df.columns)); df.head(3)

## Test 1 — inject_anomalies

In [ ]:
corr, gt = inject_anomalies(df, 'Amount Paid', rate=0.02, multiplier=50, seed=42)
idx = gt[gt].index[:5]
print('flagged:', gt.sum(), '| expected:', int(len(df)*0.02))
print('x50 ok:', np.allclose(corr.loc[idx,'Amount Paid'], df.loc[idx,'Amount Paid']*50))

## Test 2 — inject_missing_values

In [ ]:
corr2, gt2 = inject_missing_values(df, 'Receiving Currency', rate=0.10, seed=42)
n=corr2['Receiving Currency'].isna().sum()
print('NaNs:', n, '| expected:', int(len(df)*0.10), '| match:', n==int(len(df)*0.10))

## Test 3 — inject_schema_drift

In [ ]:
d1,l1=inject_schema_drift(df, drop_column='Payment Format')
d2,l2=inject_schema_drift(df, rename_map={'Amount Paid':'amt_paid'})
d3,l3=inject_schema_drift(df, dtype_change=('From Bank', str))
print('drop:', 'Payment Format' not in d1.columns, l1)
print('rename:', 'amt_paid' in d2.columns, l2)
print('dtype:', d3['From Bank'].dtype, l3)

## Test 4 — reproducibility

In [ ]:
_,a=inject_anomalies(df,'Amount Paid',seed=42)
_,b=inject_anomalies(df,'Amount Paid',seed=42)
print('same seed identical:', a.equals(b))